In [2]:
#Complete Python Program
import os
import sqlite3

# Step 1: Create Database
db_name='login_analysis.db'
if os.path.exists(db_name):
    os.remove(db_name)

conn=sqlite3.connect(db_name)
cursor=conn.cursor()

# Step 2: Create Table
cursor.execute('''
CREATE TABLE logins(
    user_id INTEGER,
    login_date TEXT
)
''')

# Step 3: Insert Sample Data
logins=[
(101,'2026-08-01'),
(101,'2026-08-02'),
(101,'2026-08-03'),
(101,'2026-08-05'),
(102,'2026-08-01'),
(102,'2026-08-03'),
(102,'2026-08-04'),
(102,'2026-08-05'),
(103,'2026-08-10'),
(103,'2026-08-11'),
(104,'2026-08-15')
]

cursor.executemany("INSERT INTO logins VALUES (?,?)",logins)
conn.commit()

print("LOGIN RECORDS")
for row in cursor.execute("SELECT * FROM logins ORDER BY user_id,login_date"):
    print(row)

query='''
WITH d AS
(
SELECT DISTINCT
user_id,
date(login_date) AS dt
FROM logins
),
g AS
(
SELECT
user_id,
dt,
julianday(dt)-ROW_NUMBER() OVER(
PARTITION BY user_id
ORDER BY dt
) AS grp
FROM d
)
SELECT
user_id,
MIN(dt) AS start_date,
MAX(dt) AS end_date,
COUNT(*) AS days
FROM g
GROUP BY user_id,grp
HAVING COUNT(*)>=3
ORDER BY user_id;
'''

print("\nUSERS WITH 3+ CONSECUTIVE LOGIN DAYS")
for row in cursor.execute(query):
    print(row)

conn.close()



LOGIN RECORDS
(101, '2026-08-01')
(101, '2026-08-02')
(101, '2026-08-03')
(101, '2026-08-05')
(102, '2026-08-01')
(102, '2026-08-03')
(102, '2026-08-04')
(102, '2026-08-05')
(103, '2026-08-10')
(103, '2026-08-11')
(104, '2026-08-15')

USERS WITH 3+ CONSECUTIVE LOGIN DAYS
(101, '2026-08-01', '2026-08-03', 3)
(102, '2026-08-03', '2026-08-05', 3)
